# LotL Guard Data Exploration
This notebook inspects both the raw JSONL (`data/dataset.jsonl`) and the validated dataset (`artifacts/processed.parquet`) to understand label balance, LOLBin usage, and potential leakage patterns before feature engineering.

Run `make preprocess` first to ensure the parquet + splits exist.

## Raw dataset snapshot (pre-validation)

In [ ]:
import json
from pathlib import Path

RAW_PATH = Path("data/dataset.jsonl")
raw_records = [json.loads(line) for line in RAW_PATH.read_text().splitlines() if line.strip()]
len(raw_records)

In [ ]:
import pandas as pd
raw_df = pd.DataFrame(raw_records)
raw_df[["_label", "claude-sonnet-4-5.predicted_label"]].head()

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline
DATA_PATH = Path('artifacts/processed.parquet')
df = pd.read_parquet(DATA_PATH)
print(df.shape)
df.head()

## Label distribution

In [ ]:
label_counts = df['_label'].value_counts().rename(index={0: 'benign', 1: 'malicious'})
label_counts

In [ ]:
label_counts.plot(kind='bar', title='Label counts')
plt.ylabel('rows')

## Top SourceImage processes

In [ ]:
df['SourceImage'].str.lower().value_counts().head(15)

## Group key sizes

In [ ]:
group_sizes = df.groupby('group_key')['row_id'].count().sort_values(ascending=False)
group_sizes.head(15)

## Keyword flags (placeholder)
Add cells here to look for encoded commands, downloads, etc.

## Ground truth `_label` vs Claude predicted label

In [ ]:
if '_label' in raw_df.columns and 'claude-sonnet-4-5.predicted_label' in raw_df.columns:
    comp = raw_df[['_label', 'claude-sonnet-4-5.predicted_label']].copy()
    comp['_label_norm'] = comp['_label'].astype(str).str.lower()
    comp['claude_norm'] = comp['claude-sonnet-4-5.predicted_label'].astype(str).str.lower()
    display(pd.crosstab(comp['_label_norm'], comp['claude_norm']))
else:
    print('Columns missing in raw_df')


## Feature distribution by Claude label

In [ ]:
df['claude_label'] = df['claude-sonnet-4-5.predicted_label'].astype(str).str.lower()
top_images = (df.groupby('claude_label')['SourceImage']
                .apply(lambda s: s.str.lower().value_counts().head(5))
                .unstack(fill_value=0))
top_images

In [ ]:
df['cmd_length'] = df['CommandLine'].fillna('').str.len()
df.boxplot(column='cmd_length', by='claude_label')
plt.title('Command length by Claude label')
plt.suptitle('')
plt.xlabel('claude label')
plt.ylabel('characters')